In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from pprint import pprint
from typing import TypedDict

from sqlalchemy.ext.asyncio import result

from src.rag import ( MultimodalRetriever, load_documents, prepare_rag_prompt)
import re
from collections import  Counter
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\agentic\Property-Rental-Management-System


In [3]:
#Shared agent state
class AgentState(TypedDict, total=False):
    """Shared state passed between every note in the orchestration graph"""

    message: str
    tenant_id: str
    unit_id: str
    intent: str
    rag_context: str
    response: str
    agent_used: str
    confidence: float

print("AgentState keys: ", list(AgentState.__annotations__.keys()))

AgentState keys:  ['message', 'tenant_id', 'unit_id', 'intent', 'rag_context', 'response', 'agent_used', 'confidence']


In [5]:
ASSET_DIR = PROJECT_ROOT / "notebooks" / "_generated_assets" / "06_agent_orchestration"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

docs_content = {
   "tenant_policy.txt": (
         "Unit A1 lease: monthly rent R4500, due on the 1st. "
        "Pets  allowed with written approval. Quiet hours 22:00-07:00."
         "Rooftop garden access 08:00-20:00. Parking bay 12 assigned. "
  ),
    "payment_log.txt": (
        "Tenant Bennet Dyani (unit A1) paid July rent in full R4500 on 2026-07-03 - on time."
        "Tenant Itumeleng Bedesho (unit B2) rent R4500 due on 2026-07-01 - OVERDUE as of 2026-07-06."
        "Late fee policy: R100 after 5 days, R200 after 10 days, R300 after 15 days."
    ),
    "maintenance_log.txt": (
        "Unit A1: Leaking kitchen sink - approved, technician visit 2026-07-07 10:00. "
        "Unit B2: Bedroom light not working - pending landlord approval. Priority: medium. "
        "Unit C3:Front door lock jam - scheduled 2026-07-15."
    ),
    "financial_summary.txt": (
        "Q2 2026 rental income: R36,000 across 8 units. "
        "Vacancy rate: 12.5% (1 unit vacant). "
        "Projected Q3 income: R54,000 assuming full occupancy from August. "
        "Maintenance costs Q2: R3,100. Net operating income Q2: R25,300."
    ),
}

doc_paths = []
for filename, content in docs_content.items():
    path = ASSET_DIR / filename
    path.write_text(content, encoding="utf-8")
    doc_paths.append(path)

metadata_by_source = {
    str(ASSET_DIR / "tenant_policy.txt"): {"category": "lease"},
    str(ASSET_DIR / "payment_log.txt"): {"category": "payment"},
    str(ASSET_DIR / "maintenance_log.txt"): {"category": "maintenance"},
    str(ASSET_DIR / "financial_summary.txt"): {"category": "financial"}
}

documents = load_documents(doc_paths, metadata_by_source=metadata_by_source)

retriever = MultimodalRetriever()
retriever.add_documents(documents, chunk_size=20, chunk_overlap=5)

print(f"Corpus loaded: {len(documents)} documents,  {len(retriever.chunks)} chunks")



Corpus loaded: 4 documents,  11 chunks


In [6]:
_INTENT_KEYWORDS: dict[str, list[str]] = {
    "tenant_qa": [
        "lease", "policy", "rule", "pet", "parking", "amenity", "garden",
        "quiet", "hours", "allowed", "tenant", "unit", "property",
    ],
    "payment": [
        "rent", "payment", "paid", "overdue", "late", "fee", "invoice",
        "due", "balance", "charge", "receipt",
    ],
    "maintenance": [
        "repair", "fix", "broken", "leak", "maintenance", "request",
        "technician", "schedule", "hvac", "window", "tap", "issue",
    ],
    "forecast": [
        "forecast", "income", "revenue", "predict", "projection", "vacancy",
        "occupancy", "q2", "q3", "financial", "net", "operating",
    ],
}

def _tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def route_intent(state: AgentState) -> AgentState:
    """Classify the user message and write intent + confidence into state."""
    tokens = Counter(_tokenize(state["message"]))
    scores: dict[str, float] = {}

    for intent, keywords in _INTENT_KEYWORDS.items():
        scores[intent] = sum(tokens[kw] for kw in keywords)

    best_intent = max(scores, key=lambda k: scores[k])
    total = sum(scores.values()) or 1
    confidence = round(scores[best_intent] / total, 3)

    if scores[best_intent] == 0:
        best_intent = "unknown"
        confidence = 0.0

    return {"intent": best_intent, "confidence": confidence}

#smoke test

test_messages = [
    "What is the lease for unit A1?",
    "What is the rent for unit B2?",
    "What is the maintenance cost for unit C3?",
    "What is the Q3 income forecast?",
]

for msg in test_messages:
    result = route_intent({"message": msg})
    print(f"{msg!r:55s} -> intent: {result['intent']:10s} confidence: {result['confidence']:.2f}")

'What is the lease for unit A1?'                        -> intent: tenant_qa  confidence: 1.00
'What is the rent for unit B2?'                         -> intent: tenant_qa  confidence: 0.50
'What is the maintenance cost for unit C3?'             -> intent: tenant_qa  confidence: 0.50
'What is the Q3 income forecast?'                       -> intent: forecast   confidence: 1.00
